<a href="https://colab.research.google.com/github/kkokay07/GenomicClass_on_Cloud/blob/master/ML%20in%20genomics/ML_Random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Random Forest Classification - Practical Implementation

## Brief Introduction
Random Forest is an ensemble learning method that combines multiple decision trees to improve prediction accuracy and control overfitting. Key features:
- Uses bootstrap sampling (random sampling with replacement)
- Creates multiple decision trees
- Combines predictions through voting
- Provides feature importance ranking

In [17]:
# Step 1: Import Required Libraries

# Import NumPy for numerical operations, random number generation,
# and handling arrays efficiently
import numpy as np

# Import Pandas for reading, manipulating, and analyzing tabular data
# such as CSV files containing SNP genotypes
import pandas as pd

# Import Matplotlib for creating plots and visualizations
# such as feature importance plots and confusion matrices
import matplotlib.pyplot as plt

# Import Seaborn for enhanced statistical visualizations with
# better aesthetics than base Matplotlib
import seaborn as sns

# Import train_test_split to divide the dataset into training
# and testing subsets for model evaluation
from sklearn.model_selection import train_test_split

# Import RandomForestClassifier to build a machine learning model
# for breed classification using SNP markers
from sklearn.ensemble import RandomForestClassifier

# Import confusion_matrix to evaluate classification performance
# and classification_report to obtain precision, recall, F1-score, and accuracy
from sklearn.metrics import confusion_matrix, classification_report

# Import LabelEncoder to convert breed names (e.g., BSW, HOL, JER)
# into numerical values required by machine learning algorithms
from sklearn.preprocessing import LabelEncoder

# Set a fixed random seed to ensure reproducible results
# so that data splitting and model training produce the same output each run
np.random.seed(42)

In [ ]:
# Step 2: Load and Preview Data
data = pd.read_csv('common_cancers.csv')  # Replace with your data file

print("Dataset Shape:", data.shape)
print("\nFirst few rows:")
display(data.head())

# Check for missing values
print("\nMissing Values:")
print(data.isnull().sum())

In [ ]:
# Step 3: Data Preprocessing

# Separate SNP marker data (features) from the cancer type labels (target variable)
# 'Individuals' contains the cancer classes (e.g., Lung_Cancer, etc.)
# All remaining 645 columns contain SNP genotype information coded as 1, 2, or 3
X = data.drop('Individuals', axis=1)
y = data['Individuals']

# Convert cancer type names into numerical labels required by
# the Random Forest classifier
# Example:
# Lung_Cancer -> 0
# Breast_Cancer -> 1
# Colon_Cancer -> 2
# (actual encoding depends on the dataset)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the dataset into training and testing subsets
# 80% of samples are used to train the model
# 20% are held out for independent performance evaluation
#
# random_state=42 ensures reproducible results
# stratify=y preserves the proportion of each cancer type
# in both training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display the dimensions of the resulting datasets
# Rows = samples/patients
# Columns = SNP markers
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
# Step 4: Create and Train Random Forest Model

# Random Forest is an ensemble learning algorithm that builds
# multiple decision trees and combines their predictions to improve
# classification accuracy and reduce overfitting
rf_model = RandomForestClassifier(

    # Number of decision trees in the forest
    # More trees generally improve stability and accuracy but increase computation time
    n_estimators=100,

    # Number of features (SNP markers) randomly selected for consideration
    # at each split in a decision tree
    # 'sqrt' means √(total number of features)
    # Helps increase diversity among trees and reduce overfitting
    max_features='sqrt',

    # Maximum depth of each decision tree
    # None means trees grow until all leaves are pure or cannot be split further
    # Deeper trees can capture complex patterns but may overfit
    max_depth=None,

    # Minimum number of samples required to split an internal node
    # A split will occur only if at least 2 samples are present
    min_samples_split=2,

    # Minimum number of samples required in a terminal (leaf) node
    # Prevents creation of leaves with very few samples
    min_samples_leaf=1,

    # Whether bootstrap sampling is used
    # True means each tree is trained on a random sample (with replacement)
    # from the training dataset
    bootstrap=True,

    # Fixes the random number generation process
    # Ensures identical results every time the code is run
    random_state=42,

    # Number of CPU cores used during training
    # -1 means use all available processor cores
    # Speeds up model training, especially for large SNP datasets
    n_jobs=-1
)

# Train the Random Forest model using the training dataset
# The model learns relationships between SNP genotypes and class labels
rf_model.fit(X_train, y_train)

In [ ]:
# Step 5: Make Predictions and Evaluate

# Make predictions
y_pred = rf_model.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                          target_names=label_encoder.classes_))

In [ ]:
# Step 6: Feature Importance Analysis

# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values('importance',
                                                   ascending=False)

# Plot feature importances
plt.figure(figsize=(12, 6))
sns.barplot(x='importance', y='feature',
            data=feature_importance.head(10))
plt.title('Top 10 Most Important Features')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

# Print importance values
print("\nTop 10 Feature Importances:")
print(feature_importance.head(10))

In [ ]:
# Step 7: Save Model (Optional)
import joblib

# Save the model
joblib.dump(rf_model, 'random_forest_model.joblib')

# Save the label encoder
joblib.dump(label_encoder, 'label_encoder.joblib')

print("Model and label encoder saved successfully!")

## Step 8: Predict New Samples

Now, let's load the saved model and label encoder to make predictions on new, unseen data.

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the saved model and label encoder
loaded_rf_model = joblib.load('random_forest_model.joblib')
loaded_label_encoder = joblib.load('label_encoder.joblib')

print("Model and label encoder loaded successfully!")

In [ ]:
import pandas as pd
samples = pd.read_csv("samples.csv")
predictions_encoded = loaded_rf_model.predict(samples)
predictions_decoded = loaded_label_encoder.inverse_transform(predictions_encoded)
for i, prediction in enumerate(predictions_decoded):
    print(f"Sample {i+1}: {prediction}")